In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install timm torch torchvision pillow opencv-python matplotlib



In [ ]:
MODEL_PATH = "/content/drive/MyDrive/vitamin_deficiency_efficientnet_b4_final.pth"

CLASS_NAMES = [
    "Vitamin_A",
    "Vitamin_B_Complex",
    "Vitamin_C",
    "Vitamin_D",
    "Vitamin_E",
    "Vitamin_K"
]


In [ ]:
import torch, timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "efficientnet_b4",
    pretrained=False,
    num_classes=len(CLASS_NAMES)
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print("✅ Phase-2 FINAL model loaded successfully")


✅ Phase-2 FINAL model loaded successfully


In [ ]:
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess_image(img_path):
    img = Image.open(img_path).convert("RGB")
    return transform(img).unsqueeze(0)


In [ ]:
DISEASE_MAP = {
    "Vitamin_A": [
        "Xerophthalmia (Night blindness)",
        "Keratomalacia"
    ],

    "Vitamin_B_Complex": [
        "Vitamin B12 deficiency – Megaloblastic anemia",
        "Vitamin B2 deficiency – Angular cheilitis",
        "Vitamin B3 deficiency – Pellagra",
        "Vitamin B1 deficiency – Beriberi"
    ],

    "Vitamin_C": [
        "Scurvy"
    ],

    "Vitamin_D": [
        "Rickets (children)",
        "Osteomalacia (adults)"
    ],

    "Vitamin_E": [
        "Peripheral neuropathy",
        "Hemolytic anemia"
    ],

    "Vitamin_K": [
        "Coagulation disorder",
        "Excessive bleeding"
    ]
}


In [ ]:
FOOD_MAP = {
    "Vitamin_A": [
        "Carrot", "Sweet potato", "Spinach",
        "Pumpkin", "Egg yolk", "Liver"
    ],

    "Vitamin_B_Complex": {
        "B1": ["Whole grains", "Pulses", "Nuts"],
        "B2": ["Milk", "Curd", "Eggs"],
        "B3": ["Peanuts", "Fish", "Chicken"],
        "B6": ["Banana", "Potato", "Chickpeas"],
        "B9": ["Leafy greens", "Beans", "Orange"],
        "B12": ["Eggs", "Milk", "Fish", "Meat"]
    },

    "Vitamin_C": [
        "Amla", "Guava", "Orange",
        "Lemon", "Capsicum"
    ],

    "Vitamin_D": [
        "Sunlight exposure",
        "Fatty fish",
        "Egg yolk",
        "Fortified milk"
    ],

    "Vitamin_E": [
        "Almonds",
        "Sunflower seeds",
        "Avocado",
        "Vegetable oils"
    ],

    "Vitamin_K": [
        "Spinach",
        "Broccoli",
        "Cabbage",
        "Kale"
    ]
}


In [ ]:
import torch.nn.functional as F

def predict_topk(img_path, k=3):
    x = preprocess_image(img_path).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0]

    topk = torch.topk(probs, k)

    results = []
    for idx, score in zip(topk.indices, topk.values):
        vitamin = CLASS_NAMES[idx]
        results.append({
            "vitamin": vitamin,
            "confidence": round(score.item() * 100, 2),
            "major_disease": DISEASE_MAP[vitamin],
            "recommended_foods": FOOD_MAP[vitamin]
        })
    return results


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = cv2.resize(cam, (380, 380))
        return cam / cam.max()


In [ ]:
def enrich_prediction(vitamin):
    diseases = DISEASE_MAP[vitamin]

    if vitamin == "Vitamin_B_Complex":
        foods = FOOD_MAP["Vitamin_B_Complex"]
    else:
        foods = FOOD_MAP[vitamin]

    return diseases, foods


In [ ]:
IMAGE_PATH = "/content/Blood cloth (89).jpeg"  # change this

results = predict_topk(IMAGE_PATH, k=2)

for i, r in enumerate(results, 1):
    print(f"\nPrediction {i}")
    print("Vitamin:", r["vitamin"])
    print("Confidence:", r["confidence"], "%")
    print("Major disease:", r["major_disease"])
    print("Recommended foods:", ", ".join(r["recommended_foods"]))



Prediction 1
Vitamin: Vitamin_K
Confidence: 100.0 %
Major disease: ['Coagulation disorder', 'Excessive bleeding']
Recommended foods: Spinach, Broccoli, Cabbage, Kale

Prediction 2
Vitamin: Vitamin_E
Confidence: 0.0 %
Major disease: ['Peripheral neuropathy', 'Hemolytic anemia']
Recommended foods: Almonds, Sunflower seeds, Avocado, Vegetable oils
